# Telco churn: exploratory analysis and baseline model

This notebook explores the raw Telco churn data and evaluates a reproducible Random Forest baseline. Run it from the repository root or from the `notebooks` directory. The production workflow and its configuration are documented in `docs/LAB7_PIPELINE.md`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

candidates = [Path("data/raw/churn.csv"), Path("../data/raw/churn.csv")]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/churn.csv; open this notebook from the project root or notebooks folder.")

df = pd.read_csv(data_path)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns from {data_path}")
display(df.head())

## Inspect target balance and missing values

`TotalCharges` is stored as text in the source file, so coerce it to a numeric column before checking missingness or fitting a model.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
display(df["Churn"].value_counts(dropna=False).rename("count").to_frame())
display(df.isna().sum().loc[lambda values: values > 0].rename("missing_count").to_frame())

## Train and evaluate a baseline

The split is stratified and fixed-seed. The preprocessing pipeline imputes numeric missing values, scales numeric columns, and one-hot encodes categorical columns. The customer identifier is excluded from model features.

In [ ]:
X = df.drop(columns=["Churn", "customerID"], errors="ignore")
y = df["Churn"].map({"No": 0, "Yes": 1})
if y.isna().any():
    raise ValueError("Unexpected or missing labels in the Churn column")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
numeric_columns = X_train.select_dtypes(include="number").columns
categorical_columns = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_columns),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_columns),
    ]
)

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
    )),
])
baseline.fit(X_train, y_train)
predictions = baseline.predict(X_test)
probabilities = baseline.predict_proba(X_test)[:, 1]
print(classification_report(y_test, predictions, target_names=["No churn", "Churn"], zero_division=0))
print(f"ROC-AUC: {roc_auc_score(y_test, probabilities):.4f}")

This notebook is for exploratory analysis and a baseline. The production MLOps pipeline uses the shared configuration and tracked artifacts in `config.yaml` and `src/mlops_pipeline/`.